In [1]:
from scipy.ndimage import zoom
import torch
import numpy as np

In [2]:
def resample_to_target_shape(data, target_shape):
    """
    Resample data to match the target shape
    """
    # Calculate zoom factors
    factors = (target_shape[0] / data.shape[0],
               target_shape[1] / data.shape[1],
               target_shape[2] / data.shape[2])

    # Resample using order=1 (linear interpolation) for continuous data
    resampled_data = zoom(data, factors, order=1)

    return resampled_data

In [3]:
def perform_region_occlusion_analysis(test_loader, model, device, atlas_data, region_labels, region_mapping, img_shape):
    """
    Perform occlusion analysis based on brain regions defined in the atlas
    """
    model.eval()

    print(f"Target image shape for resampling: {img_shape}")

    # Check if the atlas needs resampling
    if atlas_data.shape != img_shape:
        print(f"Resampling atlas from {atlas_data.shape} to {img_shape}")
        resampled_atlas = resample_to_target_shape(atlas_data, img_shape)
    else:
        print("Atlas already matches target shape, no resampling needed")
        resampled_atlas = atlas_data

    print(f"Resampled atlas shape: {resampled_atlas.shape}")

    # Initialize results dictionary to store effect per region
    region_occlusion_effects = {region: 0 for region in region_labels}
    region_sample_counts = {region: 0 for region in region_labels}

    print('======= Starting Region-Based Occlusion Analysis =============')

    sample_count = 0

    with torch.no_grad():
        for _, (input_img, ids, target, male) in enumerate(test_loader):
            # Print input_img shape for debugging
            print(f"Input image shape: {input_img.shape}")

            # Debugging: Print detailed shape information
            print(f"Input data type: {input_img.dtype}")

            # Get original prediction
            input_img = input_img.to(device).type(torch.FloatTensor)

            # Handle gender information if needed by model
            # if opt.model == 'ScaleDense':
            #     male_onehot = torch.unsqueeze(male, 1)
            #     male_onehot = torch.zeros(male_onehot.shape[0], 2).scatter_(1, male_onehot, 1)
            #     male_onehot = male_onehot.type(torch.FloatTensor).to(device)
            #     original_output = model(input_img, male_onehot)
            # else:
            #     original_output = model(input_img)

            original_output = model(input_img)

            # original_output = original_output.cpu().numpy()
            original_output = original_output[0].numpy()

            # Process each region one by one
            for region in region_labels:
                # Free up memory
                torch.cuda.empty_cache()

                # Create mask for this region
                region_mask = (resampled_atlas == region)

                # Skip if region is not present in the resampled atlas
                if not np.any(region_mask):
                    continue

                # Clone the original input
                masked_input = input_img.clone()

                # Move input to CPU for masking
                # cpu_input = masked_input.cpu().numpy()

                cpu_input = masked_input.numpy()

                # Create a zero array with the same shape
                zeroed_array = np.zeros_like(cpu_input)

                # Create a mask array by broadcasting the region mask
                # This safely handles all dimension arrangements
                mask_array = np.ones_like(cpu_input)

                # Apply the region mask - this is the key change
                # We're assuming the last 3 dimensions of cpu_input correspond to the 3D volume
                for i in range(cpu_input.shape[0]):  # batch dimension
                    # Create a view that can be applied to the 3D volume regardless of channel arrangement
                    mask_view = np.broadcast_to(~region_mask, cpu_input[i].shape)
                    cpu_input[i] = cpu_input[i] * mask_view

                # Move back to GPU
                masked_input = torch.from_numpy(cpu_input).to(device)

                # Get prediction for masked input
                # if opt.model == 'ScaleDense':
                #     masked_output = model(masked_input, male_onehot)
                # else:
                #     masked_output = model(masked_input)

                masked_output = model(masked_input)

                # masked_output = masked_output.cpu().numpy()
                masked_output = masked_output[0].numpy()
                print('region:- ',region_mapping[region])
                print('original_output:- ',original_output)
                print('masked_output:- ',masked_output)

                # Calculate effect for this region (difference from original)
                effect = abs(masked_output - original_output)

                # Accumulate effect for this region
                region_occlusion_effects[region] += effect.item()
                region_sample_counts[region] += 1

                # Clean up
                del masked_input, cpu_input
                if 'masked_output' in locals():
                    del masked_output
                torch.cuda.empty_cache()

            sample_count += 1
            print(f"Processed sample {sample_count}/{len(test_loader)}: {ids[0]}")

    # Average effects across samples
    for region in region_labels:
        if region_sample_counts[region] > 0:
            region_occlusion_effects[region] /= region_sample_counts[region]

    # Convert results to a structured array
    result_array = np.array([region_occlusion_effects[region] for region in region_labels])

    return result_array

In [4]:
import os
os.chdir('/home/omen/Documents/Rishabh/rp-ai-Triamese-ViT-v2')

In [5]:
import os
import torch
from model.MultiViewViT import MultiViewViT
from load_data import IMG_Folder
import torch.nn as nn
import nibabel as nib
from nilearn import datasets

In [6]:
def weights_init(w):
    classname = w.__class__.__name__
    if classname.find('Conv') != -1:
        if hasattr(w, 'weight'):
            # nn.init.kaiming_normal_(w.weight, mode='fan_out', nonlinearity='relu')
            nn.init.kaiming_normal_(w.weight, mode='fan_in', nonlinearity='leaky_relu')
        if hasattr(w, 'bias') and w.bias is not None:
                nn.init.constant_(w.bias, 0)
    if classname.find('Linear') != -1:
        if hasattr(w, 'weight'):
            torch.nn.init.xavier_normal_(w.weight)
        if hasattr(w, 'bias') and w.bias is not None:
            nn.init.constant_(w.bias, 0)
    if classname.find('BatchNorm') != -1:
        if hasattr(w, 'weight') and w.weight is not None:
            nn.init.constant_(w.weight, 1)
        if hasattr(w, 'bias') and w.bias is not None:
            nn.init.constant_(w.bias, 0)

In [7]:
model = MultiViewViT(
    image_sizes=[(91, 109), (91, 91), (109, 91)],
    patch_sizes=[(7, 7), (7, 7), (7, 7)],
    num_channals=[91, 109, 91],
    vit_args={
        'emb_dim': 768, 'mlp_dim': 3072, 'num_heads': 12,
        'num_layers': 12, 'num_classes': 1,
        'dropout_rate': 0.1, 'attn_dropout_rate': 0.0
    },
    mlp_dims=[3, 128, 256, 512, 1024, 512, 256, 128, 1]
)
model.apply(weights_init)
model = model.to("cpu")

# Load checkpoint
CheckpointPath = '/home/omen/Documents/Rishabh/rp-ai-Triamese-ViT-v2/output_dirMulti_VIT_best_model_with_harmonization.pth.tar'
checkpoint = torch.load(CheckpointPath, map_location="cpu")
state_dict = checkpoint["state_dict"]
new_state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
model.load_state_dict(new_state_dict)

<All keys matched successfully>

In [8]:
CheckpointPath = '/home/omen/Documents/Rishabh/rp-ai-Triamese-ViT-v2/output_dirMulti_VIT_best_model_with_harmonization.pth.tar'
CSVPath = '/home/omen/Documents/Nafisha/brainAge/Yale_longitudinal_dataset_inf/triamese_format.xlsx'
DataFolder =  '/home/omen/Documents/Nafisha/brainAge/Yale_preprocessed_data'
test_data = IMG_Folder(CSVPath, DataFolder)
device = "cpu"

In [9]:
valid_loader = torch.utils.data.DataLoader(test_data
                                         ,batch_size=1
                                         ,num_workers=0
                                         ,pin_memory=True
                                         ,drop_last=True
                                         )

In [12]:
# ======== Load AAL atlas ======== #
aal_atlas = datasets.fetch_atlas_aal()
atlas_filename = aal_atlas.maps
atlas_nii = nib.load(atlas_filename)
atlas_data = atlas_nii.get_fdata()
region_labels = np.unique(atlas_data)[1:]  # Exclude 0 (background)
region_mapping = {code: label for code, label in zip(region_labels, aal_atlas.labels)}

print(f"Number of regions in atlas: {len(region_labels)}")
print(f"Atlas shape: {atlas_data.shape}")

/tmp/ipykernel_351767/3018833866.py:2: DeprecationWarning: Starting in version 0.13, the default fetched mask will beAAL 3v2 instead.
  aal_atlas = datasets.fetch_atlas_aal()


[fetch_atlas_aal] Dataset found in /home/omen/nilearn_data/aal_SPM12
Number of regions in atlas: 116
Atlas shape: (91, 109, 91)


In [13]:
region_mapping

{np.float64(2001.0): 'Background',
 np.float64(2002.0): 'Precentral_L',
 np.float64(2101.0): 'Precentral_R',
 np.float64(2102.0): 'Frontal_Sup_L',
 np.float64(2111.0): 'Frontal_Sup_R',
 np.float64(2112.0): 'Frontal_Sup_Orb_L',
 np.float64(2201.0): 'Frontal_Sup_Orb_R',
 np.float64(2202.0): 'Frontal_Mid_L',
 np.float64(2211.0): 'Frontal_Mid_R',
 np.float64(2212.0): 'Frontal_Mid_Orb_L',
 np.float64(2301.0): 'Frontal_Mid_Orb_R',
 np.float64(2302.0): 'Frontal_Inf_Oper_L',
 np.float64(2311.0): 'Frontal_Inf_Oper_R',
 np.float64(2312.0): 'Frontal_Inf_Tri_L',
 np.float64(2321.0): 'Frontal_Inf_Tri_R',
 np.float64(2322.0): 'Frontal_Inf_Orb_L',
 np.float64(2331.0): 'Frontal_Inf_Orb_R',
 np.float64(2332.0): 'Rolandic_Oper_L',
 np.float64(2401.0): 'Rolandic_Oper_R',
 np.float64(2402.0): 'Supp_Motor_Area_L',
 np.float64(2501.0): 'Supp_Motor_Area_R',
 np.float64(2502.0): 'Olfactory_L',
 np.float64(2601.0): 'Olfactory_R',
 np.float64(2602.0): 'Frontal_Sup_Medial_L',
 np.float64(2611.0): 'Frontal_Sup_Me